# 6단계: 시계열분석 (IT기기) — 필수 티어: 도메인 단위 Prophet — 채 담당

**분석 단위**: 도메인 전체 월별 집계(단위1). 상품 단위(단위2, 전역 DL 모델)는 권장 티어로 별도 노트북(`06b_timeseries_dl_채.ipynb`)에서 다룬다.

**타겟 변수**: 월별 부정 리뷰 비율 (`GeneralPolarity == -1`인 리뷰 수 / 그 달 전체 리뷰 수)

**ML/DL 구분**: Prophet은 추세(trend) + 계절성(seasonality) + 변곡점(changepoint)을 통계적으로 분해하는 기법이다. 신경망이 아니다 (`docs/05_06단계_가이드.md` 참고).

**데이터**: IT기기는 SNS·쇼핑몰 출처 모두 RDate가 100% 있어서 (`docs/데이터_구조.md` 3-1절 — 생활·패션·화장품 SNS는 원본 자체에 이 필드가 없는 것과 대조적), 날짜 결측으로 인한 표본 손실이 없다.

In [1]:
import os
# Prophet 내부의 cmdstan이 임시 출력 csv에 자기 파일 경로를 주석으로 적어두는데, 이 경로에
# 사용자 홈 폴더의 한글 사용자명이 포함되면 cmdstanpy가 그 주석 줄을 UTF-8로 디코딩하다 깨진다
# (실제로 겪은 UnicodeDecodeError). cmdstan 임시 디렉토리를 한글 없는 경로로 강제 지정해서 회피.
os.makedirs(r"C:\temp\prophet_tmp", exist_ok=True)
os.environ["TEMP"] = r"C:\temp\prophet_tmp"
os.environ["TMP"] = r"C:\temp\prophet_tmp"

import matplotlib
matplotlib.use("Agg")  # 헤드리스 환경(디스플레이 없음)에서 plt.show()가 멈추지 않도록 비대화형 백엔드 사용
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

os.makedirs("../results/figures", exist_ok=True)

reviews = pd.read_parquet("../data/processed/reviews.parquet")
DOMAIN = "IT기기"
print(reviews["RDate_parsed"].notna().sum(), "/", len(reviews), "행에 RDate 있음")

sub_domain = reviews[reviews["Domain"] == DOMAIN]
print(f"{DOMAIN} RDate 있음: {sub_domain['RDate_parsed'].notna().sum()}/{len(sub_domain)} "
      f"({sub_domain['RDate_parsed'].notna().mean()*100:.1f}%)")

C:\Users\채승민\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


211804 / 225283 행에 RDate 있음
IT기기 RDate 있음: 45150/45150 (100.0%)


In [2]:
plt.rcParams["font.family"] = "Malgun Gothic"  # 한글 깨짐 방지 (Windows 기본 한글 폰트)
plt.rcParams["axes.unicode_minus"] = False

In [3]:
def monthly_negative_ratio(domain: str) -> pd.DataFrame:
    sub = reviews[(reviews["Domain"] == domain) & reviews["RDate_parsed"].notna()].copy()
    sub["month"] = sub["RDate_parsed"].dt.to_period("M").dt.to_timestamp()
    g = sub.groupby("month").agg(
        total=("review_id", "count"),
        negative=("GeneralPolarity", lambda s: (s == -1).sum()),
    )
    g["neg_ratio"] = g["negative"] / g["total"]
    return g.reset_index()

series_it = monthly_negative_ratio(DOMAIN)
print(f"[{DOMAIN}] {len(series_it)}개월치, 기간 {series_it['month'].min().date()} ~ {series_it['month'].max().date()}, "
      f"월평균 리뷰수 {series_it['total'].mean():.0f}건, 부정비율 평균 {series_it['neg_ratio'].mean()*100:.1f}%")

[IT기기] 83개월치, 기간 2016-01-01 ~ 2022-11-01, 월평균 리뷰수 544건, 부정비율 평균 9.8%


In [4]:
# 최근 데이터가 너무 적은 달(리뷰 수가 극단적으로 적은 달)이 있으면 Prophet이 잡음을 추세로 오인할 수 있으니 확인
thin_months = series_it[series_it["total"] < 10]
print(f"[{DOMAIN}] 월간 리뷰 10건 미만인 달: {len(thin_months)}개 / 전체 {len(series_it)}개월")
if len(thin_months) > 0:
    print(thin_months[["month", "total"]].to_string(index=False))

[IT기기] 월간 리뷰 10건 미만인 달: 0개 / 전체 83개월


In [5]:
def run_prophet(domain: str, s: pd.DataFrame):
    df = s[["month", "neg_ratio"]].rename(columns={"month": "ds", "neg_ratio": "y"})

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.1,  # 월별 집계라 변곡점을 너무 민감하게 잡지 않도록 기본값(0.05)보다 약간 완화
    )
    m.fit(df)

    future = m.make_future_dataframe(periods=3, freq="MS")
    forecast = m.predict(future)

    # 유의미한 변곡점: prophet이 추정한 delta(추세 기울기 변화량)가 작지 않은 것만
    deltas = m.params["delta"].mean(axis=0)
    sig_idx = [i for i, d in enumerate(deltas) if abs(d) >= 0.01]
    sig_changepoints = [(m.changepoints.iloc[i], deltas[i]) for i in sig_idx]

    fig = m.plot(forecast)
    ax = fig.gca()
    for cp, d in sig_changepoints:
        ax.axvline(cp, color="red", linestyle="--", alpha=0.6)
    ax.set_title(f"{domain} - 월별 부정비율 추세/예측 (빨간선 = 유의미한 변곡점)")
    fig.savefig(f"../results/figures/06_시계열_{domain}_채.png", dpi=120, bbox_inches="tight")
    plt.show()

    return m, forecast, sig_changepoints

## IT기기

In [6]:
m_it, fc_it, cps_it = run_prophet(DOMAIN, series_it)
print(f"유의미한 변곡점 {len(cps_it)}개:")
for cp, d in cps_it:
    # delta는 기울기 변화량이지 "부정비율이 실제로 좋아졌다/나빠졌다"가 아님 -- 상승세가 꺾여도 여전히
    # 상승 중일 수 있고, 하락세가 완만해져도 여전히 하락 중일 수 있음. 방향 해석은 그래프를 직접 보고 판단할 것.
    direction = "기울기가 위쪽으로 꺾임(+)" if d > 0 else "기울기가 아래쪽으로 꺾임(-)"
    print(f"  {cp.date()} | 기울기 변화량 {d:+.4f} | {direction}")

14:08:46 - cmdstanpy - INFO - Chain [1] start processing


14:08:46 - cmdstanpy - INFO - Chain [1] done processing


유의미한 변곡점 0개:


C:\Temp\prophet_tmp\ipykernel_19512\2779672701.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
